In [ ]:
# Check if using Google Colab
try:
    from google.colab import drive

    USE_COLAB = True

    print("Running in Google Colab environment.")
except ImportError:
    USE_COLAB = False
    print("Running in local environment.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import datetime as dt

import os

from dotenv import load_dotenv

try:
    import mne
    from mne_bids import BIDSPath, read_raw_bids, write_raw_bids
    from mne.preprocessing import annotate_break, ICA
    from mne_icalabel import label_components
    from pyprep.find_noisy_channels import NoisyChannels

except ImportError as err:
    if USE_COLAB:
        print("Libraries not found. Installing libraries...")
        !pip install mne mne-bids pyvista pyvistaqt python-picard mne-icalabel pyprep --quiet

        import mne
        from mne_bids import BIDSPath, read_raw_bids, write_raw_bids
        from mne.preprocessing import annotate_break, ICA
        from mne_icalabel import label_components
        from pyprep.find_noisy_channels import NoisyChannels

        print("Libraries loaded on colab")

    elif not USE_COLAB:
        print(f"Error importing libraries on local machine: {err}")


if not USE_COLAB:
    mne.viz.set_3d_backend("pyvistaqt")
    %matplotlib qt
elif USE_COLAB:
    mne.viz.set_3d_backend("notebook")
    %matplotlib inline


In [ ]:
if USE_COLAB:
    # Access Google Drive files

    from google.colab import drive
    import os

    # get data
    GOOGLE_ROOT = "/content/drive/MyDrive/research_data/mind-wandering/Brandmeyer/"
    drive.mount("/content/drive")
    os.chdir(GOOGLE_ROOT)

    print(f"Current Directory: {os.getcwd()}")

In [ ]:
# Set constants and paths

load_dotenv()  # Load environment variables from .env file
ROOT = os.getenv("ROOT")
if USE_COLAB:
    ROOT = GOOGLE_ROOT
RAW_DATA = ROOT
DERIVATIVES = f"{ROOT}/derivatives"
DATA_FOLDER = f"{DERIVATIVES}/noica"
PREPROC_DATA = f"{DATA_FOLDER}/preprocessed"
EPOCHED_DATA = f"{DATA_FOLDER}/epoched"

# Create directories if they don't exist
os.makedirs(DERIVATIVES, exist_ok=True)
os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(PREPROC_DATA, exist_ok=True)
os.makedirs(EPOCHED_DATA, exist_ok=True)

N_JOBS = -1

SUBJECTS = [f"{i:03d}" for i in range(1, 25)]  # '001' through '024'
SESSIONS = ["01", "02", "03"]

probe_mapping = {"128": 128, "2": 2, "4": 4, "8": 8, "0": 0}

misc_channels = {
    "EXG1": "eog",
    "EXG2": "eog",  # Horizontal blinks
    "EXG3": "eog",
    "EXG4": "eog",  # Vertical blinks
    "EXG5": "misc",
    "EXG6": "misc",  # Usually mastoids
    "EXG7": "ecg",
    "EXG8": "ecg",  # Usually heartbeat
    "GSR1": "gsr",
    "GSR2": "gsr",  # Skin Conductance
    "Erg1": "misc",
    "Erg2": "misc",  # Accelerometer/Force
    "Resp": "resp",  # Breathing
    "Plet": "misc",  # Photoplethysmogram (Pulse)
    "Temp": "misc",  # Temperature
    "Status": "misc",  # Button Presses
}

In [ ]:
def preprocess_raw_data(subject, session):
    try:
        # 1. Load BIDS Data
        bp = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            suffix="eeg",
            datatype="eeg",
            root=RAW_DATA,
        )

        mne.set_log_level("WARNING")  # suppress info
        raw = read_raw_bids(bp).load_data()
        mne.set_log_level("INFO")

        # Load corrected events file (if it exists)
        fixed_events_path = f"{RAW_DATA}sub-{subject}/ses-{session}/eeg/sub-{subject}_ses-{session}_task-meditation-fixed_events.tsv"
        if os.path.exists(fixed_events_path):
            events_df = pd.read_csv(fixed_events_path, sep="\t")
            # Convert to MNE format
            annot = mne.Annotations(
                onset=events_df["onset"],
                duration=events_df["duration"],
                description=events_df["value"].astype(str),
            )
            raw.set_annotations(annot)
        else:
            print(
                f"No fixed events file found for sub-{subject} ses-{session}. Proceeding without adding events."
            )

        # 2. Set Montage and Rename Channels
        montage = mne.channels.make_standard_montage("biosemi64")
        biosemi_labels = [f"A{i}" for i in range(1, 33)] + [
            f"B{i}" for i in range(1, 33)
        ]
        rename_map = {
            old: new
            for old, new in zip(biosemi_labels, montage.ch_names)
            if old in raw.ch_names
        }
        raw.rename_channels(rename_map)
        raw.set_channel_types(misc_channels)
        raw.set_montage(montage, on_missing="ignore")

        # # Print channel map for verification
        # print(f"--- Channel Map for {subject} ---")
        # ch_names = raw.ch_names
        # ch_types = raw.get_channel_types()
        # for name, kind in zip(ch_names, ch_types):
        #     print(f"Channel: {name:<15} | Type: {kind}")

        # 3. Automated bad channel detection
        raw_copy = (
            raw.copy()
            .crop(tmin=0, tmax=600)  # Use first 10 minutes for faster processing
            .filter(l_freq=1.0, h_freq=None, n_jobs=N_JOBS)
        )

        # Initialize and find bad channels
        # 'random_state' ensures reproducibility for your thesis
        nc = NoisyChannels(raw_copy, random_state=42)
        nc.find_all_bads()

        # Update raw.info['bads'] with the findings
        raw.info["bads"] = nc.get_bads()
        print(
            f"{raw.info['bads']} identified as bad channels for sub{subject} ses{session}."
        )

        # 4. Filter the raw data
        raw.filter(l_freq=2.0, h_freq=40.0, n_jobs=N_JOBS)
        raw.notch_filter(freqs=raw.info["line_freq"], n_jobs=N_JOBS)

        # # 5. Re-reference to Average
        # raw_for_ica = raw.copy().filter(l_freq=2.0, h_freq=40.0)
        # raw_for_ica.set_eeg_reference(
        #     "average", projection=False
        # )  # Permanent reference on the copy of raw (makes ICA run faster)

        # # 6. Automated ICA
        # # Initialize ICA. n_components=0.99 explains 99% of variance.
        # ica = ICA(
        #     n_components=0.99,
        #     method="infomax",
        #     fit_params=dict(extended=True),
        #     random_state=42,
        # )

        # # Fit ICA on the filtered/referenced copy
        # ica.fit(raw_for_ica, picks="eeg", decim=3)

        # # Use ICLabel to classify components
        # ic_labels = label_components(raw_for_ica, ica, method="iclabel")
        # labels = ic_labels["labels"]

        # # Identify components to exclude (anything NOT 'brain')
        # # Common labels: 'brain', 'muscle artifact', 'eye artifacts', 'heart beats', 'line noise', 'channel noise', 'other'
        # exclude_idx = [
        #     idx
        #     for idx, label in enumerate(labels)
        #     if label
        #     not in [
        #         "brain",
        #         "other",
        #     ]  # "other" is often safe to keep if you want to be conservative
        # ]

        # # Print num of excluded epochs in each category
        # print(
        #     f"--- ICA Component Classification for Sub {subject} Session {session} ---"
        # )
        # for label in set(labels):
        #     count = labels.count(label)
        #     print(f"\tComponent Label: {label} | Count: {count}")

        # # Apply the exclusion to the ICA object
        # ica.exclude = exclude_idx

        # ica.apply(raw)  # applies to original raw, not the copy
        # del raw_for_ica  # Free memory

        # 7. Interpolate bad channels (if any)
        if raw.info["bads"]:
            raw.interpolate_bads(reset_bads=True)

        # 8. Re-reference
        # raw.set_eeg_reference(
        #     ref_channels=["EXG5", "EXG6"]
        # )  # Use Mastoids as reference
        raw.set_eeg_reference(
            "average",
        )  # Use avg as reference

        # 9. Save preprocessed data
        preproc_path = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="preproc",  # Identifies these as preproc data
            suffix="eeg",  # Official BIDS suffix
            datatype="eeg",
            root=PREPROC_DATA,  # Point to your derivatives folder
            extension=".fif",
            # check=False,  # Allows us to define custom 'desc' tags
        )

        os.makedirs(preproc_path.fpath.parent, exist_ok=True)  # ensure directory exists
        raw.save(preproc_path.fpath, overwrite=True)  # Save epochs in BIDS format

        mb = os.path.getsize(preproc_path.fpath) // (1000**2)
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        message = f"{ts} ✅ Saved preprocessed Sub-{subject} Ses-{session} ({mb} MB)"
        print(message)
        return message

    except Exception as e:
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"{ts} ❌ Failed preprocessing Sub-{subject} Ses-{session}: {e}"
        print(message)
        return message

    finally:
        # Make derivative dataset desc
        desc = {
            "Name": "EEG Preprocessing Pipeline for Mind Wandering Detection Thesis",
            "BIDSVersion": "1.8.0",
            "DatasetType": "derivative",
        }
        with open(f"{PREPROC_DATA}/dataset_description.json", "w") as f:
            import json

            json.dump(desc, f, indent=4)


# Run the loop
log = []  # Array of messages to save to log file
for sub in SUBJECTS:
    for ses in SESSIONS:
        response = preprocess_raw_data(sub, ses)
        log.append(response)

# save log file
ts = dt.datetime.now().strftime("%Y-%m-%d-%H-%M")
with open(f"{PREPROC_DATA}/log-{ts}.txt", "a") as f:
    for log_message in log:
        f.write(f"{log_message}\n")

In [ ]:
def get_responses(raw):
    # returns a DataFrame with columns 'meditation_depth', 'mw_depth', 'fatigue' for each trial, extracted from the events in the raw data

    # Use the custom mapping to ensure column 3 is our trigger value
    events, _ = mne.events_from_annotations(raw, event_id=probe_mapping)

    # The stimulus trigger is exactly 128
    stim_code = 128

    meditation_lvls, mw_lvls, fatigue_lvls = [], [], []

    # Find indices where the probe occurs
    stim_indices = np.where(events[:, 2] == stim_code)[0]

    for i, idx in enumerate(stim_indices):
        # Window: from this probe until the next probe
        next_idx = stim_indices[i + 1] if i < len(stim_indices) - 1 else len(events)

        # Get the actual values (0, 2, 4, 8) recorded between probes
        trial_responses = events[idx + 1 : next_idx, 2]

        # Extract levels (defaults to -1 if the recording cut off)
        meditation_lvls.append(trial_responses[0] if len(trial_responses) > 0 else -1)
        mw_lvls.append(trial_responses[1] if len(trial_responses) > 1 else -1)
        fatigue_lvls.append(trial_responses[2] if len(trial_responses) > 2 else -1)

    responses = pd.DataFrame(
        {
            "meditation_depth": meditation_lvls,
            "mw_depth": mw_lvls,
            "fatigue": fatigue_lvls,
        }
    )

    return responses


def epoch_procdata(subject, session):
    try:
        # Load preproc data
        preproc_path = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="preproc",  # Identifies these as preproc data
            suffix="eeg",  # Official BIDS suffix
            datatype="eeg",
            root=PREPROC_DATA,  # Point to your derivatives folder
            extension=".fif",
        )

        raw = read_raw_bids(preproc_path).load_data()

        # Find Events
        events, event_id = mne.events_from_annotations(raw)
        probe_id = event_id.get("128")  # stimulus code
        if probe_id is None:
            raise ValueError(
                f"No 'stimulus' event found in annotations {event_id.keys()}"
            )

        # Epoching extracting ~10 seconds before each probe
        epochs = mne.Epochs(
            raw,
            events,
            event_id=probe_id,
            tmin=-10.05,
            tmax=-0.05,
            baseline=(None, None),  # Baseline from beginning to end of epoch
            preload=True,
            # reject=dict(eeg=200e-6),  # Auto-drop epochs certain lvl of noise
            reject=None,
            reject_by_annotation=True,  # Drop epochs with bad annotations
        )

        epochs.metadata = get_responses(raw)

        if len(epochs) == 0:
            raise ValueError(
                f"No epochs found for Sub {subject} Session {session} after preprocessing."
            )

        # Save preprocessed epochs
        epoch_path = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="epo",  # Identifies these as epochs
            suffix="eeg",  # Official BIDS suffix
            datatype="eeg",
            root=EPOCHED_DATA,  # Point to your derivatives folder
            extension=".fif",
            check=False,  # Allows us to define custom 'desc' tags
        )

        os.makedirs(epoch_path.fpath.parent, exist_ok=True)  # ensure directory exists
        epochs.save(epoch_path.fpath, overwrite=True)  # Save epochs in BIDS format
        mb = os.path.getsize(epoch_path.fpath) // (1000**2)

        num_epochs = len(epochs)
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        message = f"{ts} ✅ Saved epoched Sub-{subject} Ses-{session} with {num_epochs} epochs ({mb} MB)"
        print(message)
        return message
    except Exception as e:
        ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        message = f"{ts} ❌ Failed on Sub-{subject} Ses-{session}: {e}"
        print(message)
        return message

    finally:
        # Make derivative dataset desc
        desc = {
            "Name": "EEG Epoched Preprocessing Pipeline for Mind Wandering Detection Thesis",
            "BIDSVersion": "1.8.0",
            "DatasetType": "derivative",
        }
        with open(f"{EPOCHED_DATA}/dataset_description.json", "w") as f:
            import json

            json.dump(desc, f, indent=4)


# Run the loop
log = []  # Array of messages to save to log file
for sub in SUBJECTS:
    for ses in SESSIONS:
        response = epoch_procdata(sub, ses)
        log.append(response)

# save log file
ts = dt.datetime.now().strftime("%Y-%m-%d-%H-%M")
with open(f"{EPOCHED_DATA}/log-{ts}.txt", "a") as f:
    for log_message in log:
        f.write(f"{log_message}\n")

In [ ]:
%%script echo Skipping cell

# View preprocessed data


# Load & plot epoched data
def plot_epochs(subject, session):
    try:
        bp = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="epo",  # Identifies these as epochs
            suffix="eeg",  # Official BIDS suffix
            datatype="eeg",
            root=EPOCHED_DATA,  # Point to your derivatives folder
            extension=".fif",
            check=False,  # Allows us to define custom 'desc' tags
        )

        epochs = mne.read_epochs(bp.fpath, preload=False, verbose="ERROR")
        epochs.plot(
            n_epochs=3,
            n_channels=epochs.info["nchan"],
            scalings="auto",
            title="Cleaned 10s Epochs",
        )

        print(f"✅ Successfully loaded and plotted Sub {subject} Session {session}")
    except Exception as e:
        print(f"❌ Failed to load/plot Sub {subject} Session {session}: {e}")


def read_epochs(subject, session):
    try:
        bp = BIDSPath(
            subject=subject,
            session=session,
            task="meditation",
            description="epo",  # Identifies these as epochs
            suffix="eeg",  # Official BIDS suffix
            datatype="eeg",
            root=EPOCHED_DATA,  # Point to your derivatives folder
            extension=".fif",
            check=False,  # Allows us to define custom 'desc' tags
        )
        epochs = mne.read_epochs(bp.fpath, verbose="ERROR")

        print(epochs)
        print(epochs.info)
        print(
            f"✅ Successfully loaded Sub {subject} Session {session} with {len(epochs)} epochs"
        )
        # print epoch probe data
        # print("Epoch Metadata (Probe Responses):")
        # print(epochs.metadata)
    except Exception as e:
        print(f"❌ Failed to load Sub {subject} Session {session}: {e}")


# Run the loop
for sub in SUBJECTS:
    for ses in SESSIONS:
        plot_epochs(sub, ses)
        read_epochs(sub, ses)

In [ ]:
def get_responses_brokendata(raw):
    # This is the old function that was used to extract responses that had missing triggers and other issues.
    # It is kept here for reference but will not be used for the final analysis

    # returns a DataFrame with columns 'meditation_depth', 'mw_depth', 'fatigue' for each trial, extracted from the events in the raw data

    events, event_id = mne.events_from_annotations(raw)
    stim_code = event_id["stimulus"]

    meditation_lvls = []
    mw_lvls = []
    fatigue_lvls = []

    # Find indices where the stimulus occurs
    stim_indices = np.where(events[:, 2] == stim_code)[0]

    for idx in stim_indices:
        next_stim_idx = (
            stim_indices[stim_indices > idx][0]
            if len(stim_indices[stim_indices > idx]) > 0
            else len(events)
        )

        subsequent_events = events[
            idx + 1 : next_stim_idx
        ]  # Next key presses until nxt stimulus
        trial_responses = [ev[2] for ev in subsequent_events]

        # Map the codes to scores (defaulting to -1 if a trigger is missing/glitched)
        r1 = trial_responses[0] if len(trial_responses) > 0 else -1
        r2 = trial_responses[1] if len(trial_responses) > 1 else -1
        r3 = trial_responses[2] if len(trial_responses) > 2 else -1

        meditation_lvls.append(r1)
        mw_lvls.append(r2)
        fatigue_lvls.append(r3)

    # Create a comprehensive Metadata DataFrame
    responses = pd.DataFrame(
        {
            "meditation_depth": meditation_lvls,
            "mw_depth": mw_lvls,
            "fatigue": fatigue_lvls,
        }
    )

    return responses